# 🎓 Tutor Virtual Inteligente

Un tutor virtual basado en modelos de lenguaje de gran escala (LLM) que responde preguntas, explica conceptos y guía a estudiantes a través de tareas difíciles en múltiples materias.

**API utilizada:** [Groq](https://console.groq.com/) — acceso gratuito a modelos como `llama-3.1-8b-instant`, `llama-3.3-70b-versatile` y `gemma2-9b-it`.

---

## ¿Cómo funciona?

El sistema se apoya en tres pilares:

| Componente | Descripción |
|---|---|
| **Groq API** | Proveedor de inferencia LLM con free tier generoso (~14,400 req/día) |
| **System prompt adaptativo** | Cada materia tiene un prompt de sistema especializado que define el rol y estilo del tutor |
| **Historial de conversación** | Los mensajes se acumulan para dar contexto al modelo en cada respuesta |

### Arquitectura de mensajes

Cada llamada a la API incluye el historial completo con el siguiente formato:

```
[sistema]   → "Eres un tutor experto en Matemáticas..."
[usuario]   → "¿Qué es una derivada?"
[asistente] → "Una derivada mide la tasa de cambio..."
[usuario]   → "Dame un ejemplo"   ← nueva pregunta
```

Esto permite conversaciones contextuales donde el tutor recuerda lo que se habló anteriormente.

---

> **Para usar el tutor de forma interactiva** tienes dos opciones:
> - **UI gráfica:** `python tutor_virtual_ui.py` (recomendado)
> - **Terminal:** `python tutor_virtual.py`
>
> Este notebook sirve como explicación del código y demostración programática.

## 1. Dependencias

Este proyecto requiere únicamente la librería oficial de Groq y Jupyter:

```
groq>=0.9.0          # Cliente oficial de la API de Groq
ipywidgets>=8.1.0    # Widgets interactivos para Jupyter (referencia)
jupyterlab>=4.0.0    # Entorno Jupyter Lab
notebook>=7.0.0      # Soporte de notebooks clásico
```

Instálalas con:
```bash
pip install -r requirements.txt
```

## 2. Importaciones y configuración de la API Key

La clave de Groq se lee desde la variable de entorno `GROQ_API_KEY`.  
Obtén tu clave gratuita en: https://console.groq.com/keys

```bash
export GROQ_API_KEY="gsk_..."
```

In [6]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

# Carga variables desde el archivo .env (si existe)
load_dotenv()

api_key = os.environ.get("GROQ_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "No se encontró GROQ_API_KEY.\n"
        "Opciones:\n"
        "  1. Edita el archivo .env y reemplaza 'tu_clave_aqui' con tu clave real.\n"
        "  2. Exporta la variable: export GROQ_API_KEY='gsk_...'\n"
        "Obtén tu clave gratuita en: https://console.groq.com/keys"
    )

print("✅ API Key cargada correctamente.")

✅ API Key cargada correctamente.


## 3. Clase `TutorVirtual`

La clase encapsula toda la lógica del tutor:

- **`SYSTEM_PROMPTS`** — diccionario con un prompt de sistema para cada materia. Estos prompts definen la personalidad, el estilo explicativo y el área de conocimiento del tutor.
- **`historial`** — lista de mensajes con roles `system`, `user` y `assistant`. Se envía completa en cada llamada para mantener contexto.
- **`preguntar()`** — agrega la pregunta al historial, llama a la API y almacena la respuesta.
- **`reset_historial()`** — reinicia la conversación manteniendo solo el mensaje de sistema.

### Modelos disponibles (free tier de Groq)

| Modelo | Mejor para |
|---|---|
| `llama-3.1-8b-instant` | Respuestas rápidas, uso general |
| `llama-3.3-70b-versatile` | Razonamiento complejo, respuestas más elaboradas |
| `llama-3.1-70b-versatile` | Alternativa estable al modelo 70B |
| `gemma2-9b-it` | Modelo Google, buen rendimiento general |

In [7]:
class TutorVirtual:
    """Tutor virtual inteligente basado en Groq LLM."""

    SYSTEM_PROMPTS = {
        "General": (
            "Eres un tutor virtual amigable y paciente. Tu objetivo es ayudar al estudiante "
            "a comprender conceptos de cualquier área del conocimiento. Explica de forma clara, "
            "usa ejemplos prácticos, y si el estudiante comete un error, corrígelo con amabilidad. "
            "Responde siempre en el idioma en que el estudiante te escriba."
        ),
        "Matemáticas": (
            "Eres un tutor experto en matemáticas. Explica conceptos paso a paso, muestra "
            "el procedimiento completo para resolver problemas, y usa notación clara. "
            "Cuando el estudiante se equivoque, identifica exactamente dónde está el error. "
            "Responde siempre en el idioma en que el estudiante te escriba."
        ),
        "Programación": (
            "Eres un tutor experto en programación y ciencias de la computación. "
            "Explica conceptos con ejemplos de código comentados, sugiere buenas prácticas "
            "y ayuda a depurar errores. Usa bloques de código cuando sea apropiado. "
            "Responde siempre en el idioma en que el estudiante te escriba."
        ),
        "Ciencias": (
            "Eres un tutor experto en ciencias naturales (física, química, biología). "
            "Explica fenómenos con analogías del mundo real, conecta la teoría con la práctica, "
            "y fomenta el pensamiento crítico. "
            "Responde siempre en el idioma en que el estudiante te escriba."
        ),
        "Historia y Humanidades": (
            "Eres un tutor experto en historia, filosofía, literatura y humanidades. "
            "Contextualiza los hechos históricos, analiza textos con profundidad "
            "y ayuda al estudiante a desarrollar argumentos sólidos. "
            "Responde siempre en el idioma en que el estudiante te escriba."
        ),
        "Idiomas": (
            "Eres un tutor de idiomas. Corriges gramática y ortografía con explicaciones, "
            "enseñas vocabulario en contexto, y adaptas las lecciones al nivel del estudiante. "
            "Responde siempre en el idioma en que el estudiante te escriba."
        ),
    }

    MODELOS = [
        "llama-3.1-8b-instant",    # Rápido, uso general
        "llama-3.3-70b-versatile", # Potente, razonamiento complejo
        "llama-3.1-70b-versatile", # Alternativa estable
        "gemma2-9b-it",            # Modelo Google
    ]

    def __init__(self, api_key: str, materia: str = "General", modelo: str = "llama-3.1-8b-instant"):
        if not api_key:
            raise ValueError("Se requiere una API Key de Groq.")
        self.client = Groq(api_key=api_key)
        self.materia = materia
        self.modelo = modelo
        self.historial: list[dict] = []
        self.reset_historial()

    def reset_historial(self):
        """Reinicia el historial de conversación."""
        self.historial = [
            {"role": "system", "content": self.SYSTEM_PROMPTS.get(self.materia, self.SYSTEM_PROMPTS["General"])}
        ]

    def cambiar_materia(self, materia: str):
        """Cambia la materia y reinicia la conversación."""
        self.materia = materia
        self.reset_historial()

    def cambiar_modelo(self, modelo: str):
        """Cambia el modelo LLM sin perder el historial."""
        self.modelo = modelo

    def preguntar(self, pregunta: str, temperatura: float = 0.7, max_tokens: int = 1024) -> str:
        """Envía una pregunta al tutor y obtiene la respuesta."""
        if not pregunta.strip():
            return "Por favor escribe una pregunta."
        self.historial.append({"role": "user", "content": pregunta})
        response = self.client.chat.completions.create(
            model=self.modelo,
            messages=self.historial,
            temperature=temperatura,
            max_tokens=max_tokens,
        )
        respuesta = response.choices[0].message.content
        self.historial.append({"role": "assistant", "content": respuesta})
        return respuesta

    def historial_legible(self) -> list[dict]:
        """Retorna el historial sin el mensaje de sistema."""
        return [m for m in self.historial if m["role"] != "system"]

    def guardar_historial(self, ruta: str = "historial_sesion.json"):
        """Guarda el historial en un archivo JSON."""
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(self.historial_legible(), f, ensure_ascii=False, indent=2)
        return ruta

print("✅ Clase TutorVirtual definida.")

✅ Clase TutorVirtual definida.


## 4. Modos de uso interactivo

El proyecto incluye dos interfaces listas para usar: una con ventana gráfica y otra en terminal.

### Opción A — Interfaz gráfica: `tutor_virtual_ui.py` (recomendado)

```bash
python tutor_virtual_ui.py
```

Abre una ventana de escritorio con:

| Elemento | Descripción |
|---|---|
| **Selector de materia** | Dropdown en el header — cambia el rol del tutor al instante |
| **Selector de modelo** | Elige entre los 4 modelos de Groq sin reiniciar |
| **Chat con burbujas** | Tus mensajes a la derecha, el tutor con avatar 🤖 a la izquierda |
| **Indicador de carga** | Spinner animado mientras el modelo genera la respuesta |
| **Botón reset** 🔄 | Inicia una sesión nueva manteniendo la misma configuración |
| **Botón guardar** 💾 | Exporta el historial a `historial_sesion.json` |

Si no detecta `GROQ_API_KEY` en el archivo `.env`, aparece automáticamente un diálogo para ingresarla.

---

### Opción B — Chat en terminal: `tutor_virtual.py`

Para una experiencia sin dependencias de UI, el proyecto también incluye una interfaz de línea de comandos completa.

### Ejecución

```bash
# Con variable de entorno configurada:

python tutor_virtual.py| `/salir` | Termina el programa |

| `/guardar` | Exporta el historial a `historial_sesion.json` |

# O pasando la clave en línea:| `/historial` | Imprime los mensajes de la sesión |

GROQ_API_KEY="gsk_..." python tutor_virtual.py| `/nuevo` | Reinicia el historial |

```| `/modelo` | Cambia el modelo LLM |

| `/materia` | Cambia la materia (reinicia historial) |

### Flujo de la sesión| `/ayuda` | Muestra la lista de comandos |

|---|---|

```| Comando | Acción |

1. Selección de materia  → General | Matemáticas | Programación | ...

2. Selección de modelo   → llama3-8b | llama3-70b | mixtral | gemma2### Comandos disponibles dentro del chat

3. Chat libre en terminal
```

## 5. Demostración programática

Las siguientes celdas muestran cómo usar la clase `TutorVirtual` directamente desde Python.  
Esto ilustra la capacidad de **conversación multi-turno** con contexto acumulado.

### Ejemplo 1 — Tutor de Matemáticas (conversación contextual)

In [8]:
from IPython.display import display, Markdown

# Inicializar tutor de Matemáticas con el modelo rápido
tutor_mat = TutorVirtual(api_key=api_key, materia="Matemáticas", modelo="llama-3.1-8b-instant")

# Conversación multi-turno: cada pregunta tiene el contexto de las anteriores
preguntas = [
    "¿Qué es una derivada y para qué sirve?",
    "Dame un ejemplo concreto con f(x) = x²",
    "¿Y cuál sería la derivada de f(x) = 3x³ + 2x?",
]

for i, pregunta in enumerate(preguntas, 1):
    print(f"\n{'='*60}")
    display(Markdown(f"**🧑‍🎓 Turno {i} — Estudiante:** {pregunta}"))
    respuesta = tutor_mat.preguntar(pregunta)
    display(Markdown(f"**🤖 Tutor:**\n\n{respuesta}"))

**🧑‍🎓 Turno 1 — Estudiante:** ¿Qué es una derivada y para qué sirve?

**🤖 Tutor:**

¡Excelente pregunta!

Una derivada es una medida matemática que describe la tasa de cambio de una función con respecto a una variable independiente. En otras palabras, mide cómo cambia el valor de una función cuando cambia la variable independiente.

Imagine que estás conduciendo un coche y quieres saber a qué velocidad estás viajando. Si sabes cuánto distancia recorres en un determinado período de tiempo, puedes calcular tu velocidad. De manera similar, la derivada nos da la tasa de cambio de una función en un punto específico.

La notación matemática para la derivada de una función f(x) es:

f'(x) = lim(h → 0) [f(x + h) - f(x)] / h

Donde f(x + h) es el valor de la función en un punto cercano a x, y h es un pequeño cambio en x.

La derivada se utiliza en una variedad de campos, como:

1. **Física**: Para describir la aceleración de un objeto, la velocidad de un proceso químico, etc.
2. **Economía**: Para calcular la tasa de crecimiento de una economía, la velocidad de cambio de los precios, etc.
3. **Ingeniería**: Para diseñar sistemas y sistemas de control, optimizar procesos, etc.
4. **Ciencias naturales**: Para estudiar la evolución de las poblaciones, la temperatura en un lugar determinado, etc.

Algunas de las aplicaciones prácticas de las derivadas incluyen:

* **Optimización**: Encontrar el valor máximo o mínimo de una función.
* **Modelado de sistemas**: Describir el comportamiento de sistemas complejos.
* **Análisis de datos**: Interpretar y analizar datos para tomar decisiones informadas.

En resumen, la derivada es una herramienta fundamental en matemáticas y ciencias que nos permite describir la tasa de cambio de una función y aplicarla a una variedad de campos.

¿Tienes alguna pregunta específica sobre derivadas o necesitas ayuda para resolver un problema?

**🧑‍🎓 Turno 2 — Estudiante:** Dame un ejemplo concreto con f(x) = x²

**🤖 Tutor:**

¡Claro! Veamos un ejemplo concreto con la función f(x) = x².

**Ejemplo:** Encontrar la derivada de f(x) = x² en el punto x = 3.

**Paso 1:** Escribir la fórmula de la derivada:

f'(x) = lim(h → 0) [f(x + h) - f(x)] / h

**Paso 2:** Sustituir la función f(x) = x² en la fórmula:

f'(x) = lim(h → 0) [(x + h)² - x²] / h

**Paso 3:** Expanding la expresión (x + h)²:

f'(x) = lim(h → 0) [x² + 2xh + h² - x²] / h

**Paso 4:** Simplificar la expresión:

f'(x) = lim(h → 0) [2xh + h²] / h

**Paso 5:** Cancelar el h en el numerador y el denominador:

f'(x) = lim(h → 0) 2x + h

**Paso 6:** Evaluar el límite:

f'(x) = 2x

**Paso 7:** Ahora que tenemos la derivada, podemos encontrar el valor de la derivada en x = 3:

f'(3) = 2(3)
f'(3) = 6

**Respuesta:** La derivada de f(x) = x² en el punto x = 3 es f'(3) = 6.

**Interpretación:** Esto significa que la función f(x) = x² aumenta a una velocidad constante de 6 unidades por unidad de cambio en x en el punto x = 3.

¿Tienes alguna pregunta sobre este ejemplo o necesitas ayuda para resolver un problema similar?

**🧑‍🎓 Turno 3 — Estudiante:** ¿Y cuál sería la derivada de f(x) = 3x³ + 2x?

**🤖 Tutor:**

¡Excelente pregunta!

Para encontrar la derivada de f(x) = 3x³ + 2x, debemos aplicar la regla del producto y la suma.

**Regla del producto:** Si f(x) = u(x)v(x), entonces f'(x) = u'(x)v(x) + u(x)v'(x).

**Regla de la suma:** Si f(x) = u(x) + v(x), entonces f'(x) = u'(x) + v'(x).

En este caso, tenemos:

f(x) = 3x³ + 2x

Podemos escribir esto como:

f(x) = 3x³ + 0x² + 2x

Ahora, aplicamos la regla del producto y la suma:

f'(x) = d(3x³)/dx + d(2x)/dx
f'(x) = 3(3x²) + 2(1)
f'(x) = 9x² + 2

**Respuesta:** La derivada de f(x) = 3x³ + 2x es f'(x) = 9x² + 2.

**Interpretación:** Esto significa que la función f(x) = 3x³ + 2x aumenta a una velocidad que depende del valor de x. En particular, la función aumenta más rápido a medida que x aumenta, ya que el término 9x² es positivo y aumenta con x.

¿Tienes alguna pregunta sobre este ejemplo o necesitas ayuda para resolver un problema similar?

### Ejemplo 2 — Tutor de Programación

In [9]:
tutor_prog = TutorVirtual(api_key=api_key, materia="Programación", modelo="llama-3.1-8b-instant")

preguntas_prog = [
    "¿Qué diferencia hay entre una lista y una tupla en Python?",
    "¿Cuándo me conviene usar cada una?",
]

for i, pregunta in enumerate(preguntas_prog, 1):
    print(f"\n{'='*60}")
    display(Markdown(f"**🧑‍🎓 Turno {i} — Estudiante:** {pregunta}"))
    respuesta = tutor_prog.preguntar(pregunta)
    display(Markdown(f"**🤖 Tutor:**\n\n{respuesta}"))

**🧑‍🎓 Turno 1 — Estudiante:** ¿Qué diferencia hay entre una lista y una tupla en Python?

**🤖 Tutor:**

En Python, tanto listas como tuplas son tipos de secuencias, pero tienen algunas diferencias importantes.

**Listas**
```python
# Ejemplo de lista
mi_lista = [1, 2, 3, 4, 5]
print(mi_lista[0])  # Imprime el primer elemento de la lista
mi_lista[0] = 10  # Puedo modificar el primer elemento de la lista
print(mi_lista)  # Imprime la lista modificada
```
Las listas en Python son mutables, lo que significa que se pueden modificar después de crearlas. Las listas se pueden agregar o eliminar elementos, y sus elementos individuales se pueden modificar.

**Tuplas**
```python
# Ejemplo de tupla
mi_tupla = (1, 2, 3, 4, 5)
print(mi_tupla[0])  # Imprime el primer elemento de la tupla
# mi_tupla[0] = 10  # Error: Las tuplas no se pueden modificar después de crearlas
```
Las tuplas en Python son inmutables, lo que significa que no se pueden modificar después de crearlas. Las tuplas se pueden crear vacías o con un número determinado de elementos.

**Diferencias clave**

1. **Mutabilidad**: Las listas son mutables, mientras que las tuplas son inmutables.
2. **Modificación de elementos**: Las listas permiten modificar sus elementos individuales, mientras que las tuplas no.
3. **Uso**: Las listas se utilizan cuando se necesita modificar la estructura de datos después de crearla, mientras que las tuplas se utilizan cuando se necesita una estructura de datos inmutable y eficiente.

En resumen, si necesita una estructura de datos que se pueda modificar después de crearla, utilice una lista. Si necesita una estructura de datos inmutable y eficiente, utilice una tupla.

**🧑‍🎓 Turno 2 — Estudiante:** ¿Cuándo me conviene usar cada una?

**🤖 Tutor:**

Aquí te presento algunas situaciones en las que se prefieren listas o tuplas:

**Usar listas:**

1. **Cuando se necesita modificar la estructura de datos**: Si se necesita agregar, eliminar o modificar elementos en la estructura de datos después de crearla, se debe utilizar una lista.
2. **Cuando se necesita acceso aleatorio**: Si se necesita acceder a elementos en cualquier posición de la estructura de datos, se debe utilizar una lista.
3. **Cuando se necesita utilizar métodos adicionales**: Las listas ofrecen varios métodos adicionales, como `append()`, `insert()`, `remove()`, etc., que no están disponibles en las tuplas.
4. **Cuando se necesita almacenar datos con una estructura dinámica**: Si se necesita almacenar datos con una estructura que cambie durante la ejecución del programa, se debe utilizar una lista.

**Usar tuplas:**

1. **Cuando se necesita una estructura de datos inmutable**: Si se necesita una estructura de datos que no se pueda modificar después de crearla, se debe utilizar una tupla.
2. **Cuando se necesita una estructura de datos eficiente**: Las tuplas son más eficientes en términos de espacio de memoria y velocidad de acceso que las listas.
3. **Cuando se necesita almacenar datos con una estructura fija**: Si se necesita almacenar datos con una estructura que no cambie durante la ejecución del programa, se debe utilizar una tupla.
4. **Cuando se necesita utilizar en contextos de programación funcional**: Las tuplas se utilizan a menudo en programación funcional, donde se necesitan estructuras de datos inmutables.

**Ejemplos de uso:**

* **Lista para almacenar un conjunto de elementos dinámicos**:
```python
mi_lista = []
while True:
    elemento = input("Ingrese un elemento: ")
    if elemento == "":
        break
    mi_lista.append(elemento)
print(mi_lista)
```
* **Tupla para almacenar un conjunto de elementos fijos**:
```python
mi_tupla = ("Hola", "Mundo", "Python")
print(mi_tupla)
```
Recuerda que, en general, si no se necesita modificar la estructura de datos después de crearla, es preferible utilizar una tupla.

## 6. Exportar historial de la sesión

El historial de cualquier sesión se puede exportar a JSON para revisión posterior o análisis.

In [10]:
# Exportar historial del tutor de matemáticas
ruta = tutor_mat.guardar_historial("historial_sesion.json")
historial = tutor_mat.historial_legible()

print(f"✅ Historial guardado en '{ruta}' ({len(historial)} mensajes)\n")
print(json.dumps(historial, ensure_ascii=False, indent=2))

✅ Historial guardado en 'historial_sesion.json' (6 mensajes)

[
  {
    "role": "user",
    "content": "¿Qué es una derivada y para qué sirve?"
  },
  {
    "role": "assistant",
    "content": "¡Excelente pregunta!\n\nUna derivada es una medida matemática que describe la tasa de cambio de una función con respecto a una variable independiente. En otras palabras, mide cómo cambia el valor de una función cuando cambia la variable independiente.\n\nImagine que estás conduciendo un coche y quieres saber a qué velocidad estás viajando. Si sabes cuánto distancia recorres en un determinado período de tiempo, puedes calcular tu velocidad. De manera similar, la derivada nos da la tasa de cambio de una función en un punto específico.\n\nLa notación matemática para la derivada de una función f(x) es:\n\nf'(x) = lim(h → 0) [f(x + h) - f(x)] / h\n\nDonde f(x + h) es el valor de la función en un punto cercano a x, y h es un pequeño cambio en x.\n\nLa derivada se utiliza en una variedad de campos, com